# Artificial Intelligence Laboratory: Search and A*

## Submission Notebook

This notebook contains the implementation, execution, and tests for the warehouse search laboratory.

**The reflection/final-answer questions are intentionally not included here. They are included only in the accompanying editable Word report.**

## Task 0 — Search problem

Warehouse:

```text
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
```

A state is the robot's `(row, column)` position. Actions are Up, Down, Left, and Right. A transition moves one cell if the destination is inside the grid and not `#`. The initial state is `S`, the goal is `G`, and every movement has cost 1.

In [ ]:
import heapq
import math
from collections import deque

warehouse = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
]

ACTIONS = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1),
}

def locate(grid, symbol):
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch == symbol:
                return (r, c)
    raise ValueError(symbol)

def valid_neighbors(grid, state):
    r, c = state
    for action, (dr, dc) in ACTIONS.items():
        nxt = (r + dr, c + dc)
        if (0 <= nxt[0] < len(grid)
            and 0 <= nxt[1] < len(grid[0])
            and grid[nxt[0]][nxt[1]] != "#"):
            yield nxt, action

def reconstruct(parent, action_used, goal):
    path, actions = [], []
    current = goal
    while current is not None:
        path.append(current)
        if current in action_used:
            actions.append(action_used[current])
        current = parent[current]
    return path[::-1], actions[::-1]

In [ ]:
def heuristic(state, goal, kind="manhattan", weight=1.0):
    dr = abs(state[0] - goal[0])
    dc = abs(state[1] - goal[1])

    if kind == "manhattan":
        return dr + dc
    if kind == "zero":
        return 0
    if kind == "euclidean":
        return math.sqrt(dr**2 + dc**2)
    if kind == "scaled":
        return weight * (dr + dc)
    raise ValueError(kind)

def astar(grid, hkind="manhattan", weight=1.0):
    start, goal = locate(grid, "S"), locate(grid, "G")
    heap = [(heuristic(start, goal, hkind, weight), 0, 0, start)]
    counter = 0
    g = {start: 0}
    parent = {start: None}
    action_used = {}
    closed = set()
    expanded = 0

    while heap:
        f, current_g, _, current = heapq.heappop(heap)

        if current in closed or current_g != g.get(current):
            continue

        closed.add(current)
        expanded += 1

        if current == goal:
            path, actions = reconstruct(parent, action_used, goal)
            return {
                "found": True,
                "path": path,
                "actions": actions,
                "length": len(actions),
                "expanded": expanded
            }

        for nxt, action in valid_neighbors(grid, current):
            tentative_g = current_g + 1

            if tentative_g < g.get(nxt, float("inf")):
                g[nxt] = tentative_g
                parent[nxt] = current
                action_used[nxt] = action
                counter += 1
                fscore = tentative_g + heuristic(nxt, goal, hkind, weight)
                heapq.heappush(heap, (fscore, tentative_g, counter, nxt))

    return {
        "found": False,
        "path": None,
        "actions": None,
        "length": None,
        "expanded": expanded
    }

In [ ]:
def bfs(grid):
    start, goal = locate(grid, "S"), locate(grid, "G")
    queue = [start]
    head = 0
    parent = {start: None}
    action_used = {}
    expanded = 0

    while head < len(queue):
        current = queue[head]
        head += 1
        expanded += 1

        if current == goal:
            path, actions = reconstruct(parent, action_used, goal)
            return {
                "found": True,
                "path": path,
                "actions": actions,
                "length": len(actions),
                "expanded": expanded
            }

        for nxt, action in valid_neighbors(grid, current):
            if nxt not in parent:
                parent[nxt] = current
                action_used[nxt] = action
                queue.append(nxt)

    return {
        "found": False,
        "path": None,
        "actions": None,
        "length": None,
        "expanded": expanded
    }

In [ ]:
# Task 3 — Original warehouse

astar_result = astar(warehouse, "manhattan")
bfs_result = bfs(warehouse)

print("A* found:", astar_result["found"])
print("A* path:", astar_result["path"])
print("A* actions:", astar_result["actions"])
print("A* path length:", astar_result["length"])
print("A* states expanded:", astar_result["expanded"])
print()
print("BFS found:", bfs_result["found"])
print("BFS path length:", bfs_result["length"])
print("BFS states expanded:", bfs_result["expanded"])

In [ ]:
# Task 3 — Required tests

def validate_path(grid, result):
    assert result["found"]
    path = result["path"]
    assert path[0] == locate(grid, "S")
    assert path[-1] == locate(grid, "G")

    for r, c in path:
        assert grid[r][c] != "#"

    for a, b in zip(path, path[1:]):
        assert abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1

# 1. Original warehouse
validate_path(warehouse, astar_result)
assert astar_result["length"] == bfs_result["length"] == 40

# 2. Trivial one-step case
trivial = ["#####", "#SG##", "#####"]
trivial_result = astar(trivial, "manhattan")
assert trivial_result["found"] and trivial_result["length"] == 1

# 3. No solution
no_solution = ["#######", "#S....#", "###.###", "#...#G#", "#######"]
no_solution_result = astar(no_solution, "manhattan")
assert not no_solution_result["found"]

# 4. Alternative paths
alternative = ["#######", "#S...G#", "#.....#", "#.....#", "#######"]
alternative_result = astar(alternative, "manhattan")
assert alternative_result["found"] and alternative_result["length"] == 4
validate_path(alternative, alternative_result)

print("All required tests passed.")

In [ ]:
# Task 4 — Identify A* components

print("State: (row, column)")
print("Action: Up / Down / Left / Right")
print("Transition: valid_neighbors()")
print("Goal test: current == goal")
print("g(n): dictionary g")
print("h(n): heuristic()")
print("f(n): tentative_g + heuristic(...)")
print("Frontier: heapq priority queue")
print("Visited/closed states: closed set")
print("Path reconstruction: reconstruct()")

In [ ]:
# Task 5 — BFS versus A*

print(f"{'Measure':<20} {'BFS':>12} {'A*':>12}")
print(f"{'Solution found':<20} {str(bfs_result['found']):>12} {str(astar_result['found']):>12}")
print(f"{'Path length':<20} {bfs_result['length']:>12} {astar_result['length']:>12}")
print(f"{'States expanded':<20} {bfs_result['expanded']:>12} {astar_result['expanded']:>12}")

In [ ]:
# Task 6 — Heuristic investigation

heuristic_results = {
    "Manhattan": astar(warehouse, "manhattan"),
    "h(n)=0": astar(warehouse, "zero"),
    "Euclidean": astar(warehouse, "euclidean"),
    "2 × Manhattan": astar(warehouse, "scaled", 2.0),
}

print(f"{'Heuristic':<18} {'Found':>8} {'Path length':>14} {'States expanded':>18}")
for name, result in heuristic_results.items():
    print(f"{name:<18} {str(result['found']):>8} {result['length']:>14} {result['expanded']:>18}")